## Topic 6: Migrating the In-Memory Store into Qdrant

### Concept, Intuition, Why It Exists

- Earlier chapters built an in-memory vector store: a plain Python list of dicts, each holding `{"text":, "source":, "embedding":}`, searched by looping through every entry and computing cosine similarity one by one via `np.dot`. This was the right starting point — zero infrastructure, easy to understand, correct at demo scale.
- Three concrete problems show up the moment it needs to be more than a demo. First, **no persistence** — every process restart loses the entire store and forces a full re-embed of every chunk from scratch, which gets expensive as the corpus grows. Second, **no metadata filtering** — the only way to scope a search to, say, only FAQ documents is to loop through all results afterward and discard the non-FAQ ones, which means the search fetched too many and still might not return k useful results. Third, **brute-force scan scales linearly** — at hundreds of chunks it's invisible, at hundreds of thousands it's seconds per query.
- Migrating to Qdrant solves all three at once. The migration is deliberately minimal — the Document shape from the ingestion chapter maps directly onto Qdrant's collections/points/payloads model with no structural redesign, showing that the abstraction choices made earlier were forward-compatible with a real vector DB without needing to be rebuilt from scratch.

### What Actually Changes

- **Before (in-memory)**: embed all chunks → store in a Python list → search by looping every entry and computing `np.dot(query_vector, entry["embedding"])` → sort → return top-k.
- **After (Qdrant)**: embed all chunks → upsert as points into a Qdrant collection → search by calling `client.query_points(query=query_vector, limit=k)` → Qdrant's HNSW index handles the rest.
- The embedding step is identical. The model is identical. The chunk shape is identical. The only things that change are where vectors are stored and how nearest-neighbor search is executed.

### How It's Implemented In This Project

- The in-memory store used `{"text":, "source":, "embedding":}` dicts in a plain list.
- Qdrant replaces this with a collection of points where `text` and `source` move into the payload, and `embedding` becomes the point's vector.
- The search function changes from a manual dot-product loop to a single `client.query_points()` call.
- Everything upstream (loaders, chunker, embedding model) and everything downstream (the answer being generated from retrieved chunks) stays completely unchanged — migration only touches the storage and retrieval layer.

### Real-World Issues, Edge Cases, Debugging

- **Re-embedding on every restart was the hidden cost of the in-memory store**: at 17 pages across 4 JSON files it takes seconds. At a real NBFC corpus of thousands of pages it would take minutes or longer, making a process restart a costly event rather than a routine one. Qdrant's persistence eliminates this entirely once Docker is used (`:memory:` mode still re-embeds on restart, same as the list-based approach).
- **The old brute-force search silently degraded as the corpus grew**: nothing signalled that search was getting slower — it just gradually became measurably slower per query as more chunks were added. HNSW indexing keeps query time roughly constant regardless of corpus size.
- **Filtered search correctness**: the old approach fetched all results and filtered afterward — if 100 results came back and 95 were the wrong doc type, you might only get 5 useful results even though you asked for k=10. Qdrant's payload filtering during HNSW traversal means the k returned results already satisfy the filter, so you always get k useful results back.

### Design Decisions & Trade-offs

- Migrate to `:memory:` mode first, not straight to Docker: the code change is identical either way, but `:memory:` mode lets the migration be validated without any infrastructure setup — the correctness of the new storage and retrieval layer can be confirmed before adding the operational concern of running a Docker container.
- Keep the embedding model the same across the migration: the whole point of the migration is to change the storage and search layer without touching anything else. Changing the embedding model at the same time would make it impossible to tell whether a change in retrieval quality came from the storage migration or the model change.
- One migration, not a gradual dual-write phase: at this project's scale, running both the old in-memory store and the new Qdrant collection in parallel during migration adds complexity with no benefit. At production scale with live traffic, a gradual migration (write to both, read from old, then cut over reads to new) would be the safer pattern.

### Alternatives & When To Use Each

- Stay on in-memory if: corpus is small, process restarts are rare, no metadata filtering is needed, and persistence genuinely doesn't matter for the use case (e.g. a CI test or a one-off analysis script).
- Migrate to Qdrant `:memory:` if: metadata filtering is needed, or the brute-force scan latency has become measurable, but persistence still isn't required.
- Migrate to Qdrant with Docker if: data must survive process restarts, or the corpus is large enough that re-embedding on startup is too costly.

### Common Mistakes & Production Failures

- Migrating the storage layer and the embedding model at the same time, then not being able to diagnose whether a retrieval quality change came from one or the other.
- Forgetting that `:memory:` mode still loses all data on restart — it solves the filtering and latency problems but not the persistence problem, which only Docker or a hosted instance solves.
- Not validating that the migrated Qdrant store returns the same top results as the old in-memory store for known queries before decommissioning the old store — a sanity check that takes five minutes and catches any ID, payload, or dimension mismatch before it becomes a production issue.

### Lead-Level Interview Questions

**Q: The in-memory store was working fine. What is the concrete trigger to migrate, rather than migrating preemptively?**
A: Three concrete triggers, in order of likelihood: process restarts force a full re-embed that takes long enough to be a real operational problem; a query requirement needs metadata filtering that post-search filtering can't reliably satisfy (always getting k useful results back, not k results of which some are the wrong type); or corpus size makes brute-force scan latency measurably slow. Before any of those three are real problems, migration adds infrastructure cost with no user-visible benefit.

**Q: What stays the same across the migration and what actually changes?**
A: Everything stays the same except storage and search execution. The embedding model, the chunking strategy, the Document shape, the ingestion pipeline, and the downstream answer generation all stay identical. What changes: vectors move from a Python list to a Qdrant collection, and the search loop (`for chunk in store: np.dot(...)`) is replaced by a single `client.query_points()` call. The migration is deliberately narrow so any change in retrieval behavior is attributable to the storage layer, not to an accidental change in something else.

**Q: After migrating to Qdrant, how would you validate the migration produced correct results before decommissioning the old in-memory store?**
A: Run a small set of known queries against both the old in-memory store and the new Qdrant collection and compare the top-k results. They should return the same chunks in the same order for the same queries, since the embedding model and corpus are identical. Any divergence means a bug in the migration — a dimension mismatch, a wrong distance metric, or a payload field that didn't transfer correctly — caught before it becomes a live issue.

### Hidden Concepts Worth Knowing

- **The migration validates the Document abstraction decision from the ingestion chapter**: the fact that migrating from a plain Python list to a real vector database required changing exactly two things (where vectors are stored, how search is called) and nothing upstream or downstream is a direct payoff of the Document pattern established earlier. If the ingestion chapter had mixed format-specific logic into the storage layer, this migration would have been a rewrite, not a swap.
- **Build-then-swap for zero-downtime migration at production scale**: at this project's scale, a direct cutover is fine. In production with live traffic, the safer pattern is to build the new Qdrant collection in the background while the old store still serves queries, validate it, then atomically swap the retrieval path to point at the new collection — Qdrant's collection aliases make this a single API call.

### Revision Summary

> The in-memory store was correct and the right starting point. Migration to Qdrant is triggered by one of three concrete needs: persistence across restarts, metadata filtering that post-search filtering can't reliably satisfy, or corpus scale making brute-force scan latency measurable. What changes in the migration: storage moves from a Python list to a Qdrant collection, and the search loop is replaced by a single `client.query_points()` call. Everything else — embedding model, chunking, loaders, downstream generation — stays identical.

In [1]:
"""
qdrant_migration.py
---------------------
Side-by-side: the old in-memory vector store vs. the new Qdrant store.

Shows exactly what changes in the migration and what stays the same:
  SAME  : embedding model, chunk shape, corpus, query
  CHANGE: where vectors are stored + how nearest-neighbor search is called

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers numpy
"""

import hashlib
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue,
)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
COLLECTION_NAME = "fd_knowledge_base"
VECTOR_SIZE = 384

# load model once -- shared by both the old store and the new Qdrant store
model = SentenceTransformer(MODEL_NAME)

# in-memory Qdrant client -- same behavior as Docker, no persistence on restart
client = QdrantClient(":memory:")

# sample chunks -- same shape as chunker.py produces
CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to death of the depositor.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "What is the minimum amount to open an FD? The minimum deposit is Rs 25,000.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "Can I withdraw my FD before maturity? Yes, subject to a penalty.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "Nomination facility is available for all Fixed Deposit accounts.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "TDS is deducted at source if interest income exceeds Rs 40,000 per year.",
     "source": "FD_Policy.json", "doc_type": "policy"},
]


# ======================================================================
# OLD: in-memory store (what earlier chapters used)
# ======================================================================

def build_inmemory_store(chunks: list) -> list:
    """Embeds all chunks and stores them in a plain Python list.
    This is exactly what the earlier chapter's knowledge base was."""

    # embed all texts in one batch call
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts, normalize_embeddings=True)

    # store each chunk as a dict with its embedding attached
    store = []
    for i, chunk in enumerate(chunks):
        store.append({
            "text": chunk["text"],
            "source": chunk["source"],
            "doc_type": chunk["doc_type"],
            "embedding": embeddings[i],  # numpy array stored directly in memory
        })
    return store


def search_inmemory(store: list, query: str, k: int = 3) -> list:
    """Brute-force search: compute dot product against every entry, sort, return top-k.
    Correct at small scale, scales linearly -- gets slower as corpus grows."""

    # embed the query the same way chunks were embedded
    query_vector = model.encode(query, normalize_embeddings=True)

    # compute cosine similarity against every stored entry (normalized -> dot product)
    scores = [(np.dot(query_vector, entry["embedding"]), entry) for entry in store]

    # sort by score descending and return top-k
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:k]


def search_inmemory_filtered(store: list, query: str, doc_type: str, k: int = 3) -> list:
    """Filtered search the old way: search everything, then discard non-matching.
    Problem: if most results are the wrong doc_type, we get fewer than k useful results."""

    query_vector = model.encode(query, normalize_embeddings=True)
    scores = [(np.dot(query_vector, entry["embedding"]), entry) for entry in store]
    scores.sort(key=lambda x: x[0], reverse=True)

    # filter AFTER fetching -- may return fewer than k if most results don't match
    filtered = [(s, e) for s, e in scores if e["doc_type"] == doc_type]
    return filtered[:k]


# ======================================================================
# NEW: Qdrant store (the migration target)
# ======================================================================

def make_point_id(source: str, chunk_index: int) -> int:
    """Deterministic ID so re-upserts update in place, not duplicate."""
    key = f"{source}::{chunk_index}"
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)


def build_qdrant_store(chunks: list) -> None:
    """Embeds all chunks and upserts them into Qdrant.
    The embedding step is identical to the in-memory version."""

    # create the collection -- same as Topic 5
    existing = [c.name for c in client.get_collections().collections]
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )

    # embed all texts in one batch -- identical to the in-memory version
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts, normalize_embeddings=True)

    # build points -- vector goes to Qdrant, text/source/doc_type go to payload
    points = [
        PointStruct(
            id=make_point_id(chunks[i]["source"], i),
            vector=embeddings[i].tolist(),  # numpy array -> list for serialization
            payload={
                "text": chunks[i]["text"],          # stored for self-contained retrieval
                "source": chunks[i]["source"],       # provenance metadata
                "doc_type": chunks[i]["doc_type"],   # used for filtering
            },
        )
        for i in range(len(chunks))
    ]

    # upsert = insert new or replace existing -- idempotent
    client.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"Qdrant store: {client.get_collection(COLLECTION_NAME).points_count} points")


def search_qdrant(query: str, k: int = 3) -> None:
    """Qdrant search -- replaces the manual dot-product loop entirely.
    HNSW handles the nearest-neighbor search internally."""

    # embed the query -- identical to the in-memory version
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # one call replaces the entire for loop + sort from the in-memory version
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,     # the embedded query vector
        limit=k,                # return the top-k most similar points
        with_payload=True,      # include text/source/doc_type in results
    ).points                    # unwrap the response to get the list

    print(f"\nQdrant search: '{query}'")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")


def search_qdrant_filtered(query: str, doc_type: str, k: int = 3) -> None:
    """Qdrant filtered search -- filter applied DURING HNSW traversal.
    Always returns k results that match the filter, not k results then discard."""

    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=Filter(
            must=[FieldCondition(key="doc_type", match=MatchValue(value=doc_type))]
        ),
        limit=k,
        with_payload=True,
    ).points

    print(f"\nQdrant filtered search (doc_type='{doc_type}'): '{query}'")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")


# ======================================================================
# Run both side by side so the difference is visible
# ======================================================================

QUERY = "What happens if I close my FD early?"

print("=" * 60)
print("OLD: IN-MEMORY STORE")
print("=" * 60)

# build the old store
inmemory_store = build_inmemory_store(CHUNKS)
print(f"In-memory store: {len(inmemory_store)} entries")

# unfiltered search -- old way
print(f"\nIn-memory search: '{QUERY}'")
for score, entry in search_inmemory(inmemory_store, QUERY, k=3):
    print(f"  score={score:.4f} | [{entry['doc_type']}] {entry['text'][:60]}")

# filtered search -- old way (post-search filter, may return fewer than k)
print(f"\nIn-memory filtered search (doc_type='faq'): '{QUERY}'")
for score, entry in search_inmemory_filtered(inmemory_store, QUERY, doc_type="faq", k=3):
    print(f"  score={score:.4f} | [{entry['doc_type']}] {entry['text'][:60]}")

print("\n" + "=" * 60)
print("NEW: QDRANT STORE")
print("=" * 60)

# build the Qdrant store -- same embedding step, different storage
build_qdrant_store(CHUNKS)

# unfiltered search -- new way
search_qdrant(QUERY, k=3)

# filtered search -- new way (filter during HNSW, always returns k)
search_qdrant_filtered(QUERY, doc_type="faq", k=3)

print("\n" + "=" * 60)
print("WHAT CHANGED:")
print("  storage  : Python list  ->  Qdrant collection")
print("  search   : for loop + np.dot  ->  client.query_points()")
print("  filtering: post-search discard  ->  during HNSW traversal")
print("WHAT STAYED THE SAME:")
print("  embedding model, chunk shape, corpus, query, results")
print("=" * 60)


OLD: IN-MEMORY STORE
In-memory store: 8 entries

In-memory search: 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.4037 | [policy] This penalty does not apply if the FD is closed due to death
  score=0.3879 | [product] The FD interest rate for 24 months is 7.1 percent per annum.

In-memory filtered search (doc_type='faq'): 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.3675 | [faq] What is the minimum amount to open an FD? The minimum deposi

NEW: QDRANT STORE
Qdrant store: 8 points

Qdrant search: 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.4037 | [policy] This penalty does not apply if the FD is closed due to death
  score=0.3879 | [product] The FD interest rate for 24 months is 7.1 percent per annum.

Qdrant filtered search (d